In [3]:
import os
from dotenv import load_dotenv


#load env
load_dotenv()

# get api keys
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serpapi_api_key = os.getenv("SERPAPI_API_KEY")


In [6]:

from langchain_openai import ChatOpenAI


# LLM (OpenRouter)
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_api_key
)


# Test
resp=llm.invoke("Generate a list of 5 interesting facts about Python programming.");
print(resp);


content='Sure! Here are five interesting facts about Python programming:\n\n1. **Created in the Late 1980s**: Python was conceived in the late 1980s by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands. He aimed to create a language that was easy to read and write, making it accessible for developers of all experience levels.\n\n2. **Interpreted Language**: Unlike many programming languages that need to be compiled before execution, Python is an interpreted language. This means that Python code is executed line by line, which makes it easier to debug and experiment with code in real-time, but can lead to slower performance compared to compiled languages.\n\n3. **Massive Library Ecosystem**: Python boasts a rich ecosystem of libraries and frameworks, including popular ones like NumPy for numerical computing, pandas for data analysis, Django for web development, and TensorFlow for machine learning. This extensive collection allows developers to work efficiently 

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_community.callbacks import get_openai_callback

In [19]:
REVIEW_TEMPLATE = """
You are an expert reviewer of MCQs.

Your job is to analyze the given MCQs and ensure:

### CHECK:
- Questions are correct and meaningful
- Options are relevant
- Answer is correct
- Number of options is correct
- Format is valid JSON
- No duplicate questions

### FIX RULES:
- If any question is incorrect → fix it
- If answer is wrong → correct it
- If format is wrong → fix JSON
- If options are missing → regenerate that question

### INPUT MCQs:
{mcqs}

### OUTPUT:
Return ONLY corrected JSON in the same format.
Do not add explanation.
"""



TEMPLATE = """
You are an expert MCQ generator.

Your task is to generate {number} multiple-choice questions based on the given content.

### INPUT:
Topic/Content:
{text}

Optional Focus Areas:
{subtopics}

### REQUIREMENTS:
- Difficulty level: {difficulty}
- Question type: {question_type} (single or multiple correct answers)
- Each question must have exactly {num_options} options
- Label options alphabetically (A, B, C, D, E, ...)
- Questions must be clear, concise, and non-repetitive
- Questions must strictly be based on the provided content (no hallucination)

### ANSWER RULES:
- If question_type = "single" → ONLY ONE correct answer
- If question_type = "multiple" → MULTIPLE correct answers allowed

### OUTPUT FORMAT (STRICT JSON):
[
  {{
    "question": "string",
    "options": {{
      "A": "string",
      "B": "string",
      "C": "string",
      "D": "string"
    }},
    "answer": "A" 
    // OR ["A","C"] if multiple answers
    ,
    "type": "{question_type}",
    "difficulty": "{difficulty}"
  }}
]

Return ONLY JSON. No extra text. 
"""

In [21]:

#Prompt
prompt = PromptTemplate(
    input_variables=["number", "text", "subtopics", "difficulty", "question_type", "num_options"],
    template=TEMPLATE
)

# Generator
generator_chain=prompt | llm


# Reviewer
review_prompt=PromptTemplate(
    input_variables=["questions"],
    template=REVIEW_TEMPLATE
);


review_chain=review_prompt | llm




# Step 1: Generate MCQs
raw_response=generator_chain.invoke({
    "number": 5,
    "text": "Python programming",
    "subtopics": ["variables", "functions"],
    "difficulty": "medium",
    "question_type": "mixed multiple and single",## singel or mixed multiple and single
    "num_options": 5
});


raw_mcqs = raw_response.content



# Step 2: Review & Fix
final_response=review_chain.invoke({
    "mcqs": raw_mcqs
});


final_mcqs = final_response.content
print(final_mcqs)


```json
[
  {
    "question": "Which of the following is a valid variable name in Python?",
    "options": {
      "A": "my_variable",
      "B": "1stVariable",
      "C": "my-variable",
      "D": "my variable",
      "E": "myVariable_123"
    },
    "answer": ["A", "E"],
    "type": "mixed multiple and single",
    "difficulty": "medium"
  },
  {
    "question": "What will the following code output? \n\n```python\nx = 5\ny = '5'\nprint(x + y)\n```",
    "options": {
      "A": "5",
      "B": "55",
      "C": "TypeError",
      "D": "None",
      "E": "5.0"
    },
    "answer": "C",
    "type": "mixed multiple and single",
    "difficulty": "medium"
  },
  {
    "question": "In Python, which keywords are used to define a function?",
    "options": {
      "A": "def",
      "B": "function",
      "C": "defun",
      "D": "lambda",
      "E": "init"
    },
    "answer": ["A"],
    "type": "mixed multiple and single",
    "difficulty": "medium"
  },
  {
    "question": "What will the ou